# Cleaning_Final.ipynb

Produces three cleaned `_final` tables in `developer_project.duckdb`:
- `contact_final` — developer profiles merged with `contact-addition.csv` supplement
- `activity_final` — engagement events, deduplicated, scores capped to [0, 100]
- `sdk_download_final` — download records, deduplicated, negative counts fixed

Then creates 100k random samples of each and exports them to `Data/`.

**Run order:** `Creating_duckDB.ipynb` → `EDA_DuckDB.ipynb` → **this notebook** → `CleanDataSanity.ipynb`

## Section 0 — Setup

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("developer_project.duckdb")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print("Connected to developer_project.duckdb")
print("Existing tables:")
display(con.execute("SHOW TABLES").df())

c:\Users\ldcal\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Connected to developer_project.duckdb
Existing tables:


,name
0,activity_base
1,activity_clean
2,activity_final
3,activity_raw
4,activity_sample
5,activity_score_mapping_clean
6,activity_score_mapping_raw
7,activity_work
8,contact_clean
9,contact_clean_backup_before_supplement


In [2]:
def validate_table(con, table_name, key_col, date_col=None, score_col=None):
    """Print row count, null % on key column, duplicate count, and optionally date/score ranges."""
    row_count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    null_key = con.execute(
        f"SELECT COUNT(*) FROM {table_name} WHERE {key_col} IS NULL OR TRIM({key_col}) = ''"
    ).fetchone()[0]
    dupe_count = con.execute(
        f"SELECT COUNT(*) - COUNT(DISTINCT {key_col}) FROM {table_name}"
    ).fetchone()[0]

    print(f"\n{'='*50}")
    print(f"Table: {table_name}")
    print(f"  Rows          : {row_count:,}")
    print(f"  Null {key_col:<15}: {null_key:,} ({null_key/row_count*100:.2f}%)")
    print(f"  Duplicates on {key_col}: {dupe_count:,}")

    if date_col:
        date_range = con.execute(
            f"SELECT MIN({date_col}), MAX({date_col}) FROM {table_name}"
        ).fetchone()
        print(f"  {date_col} range: {date_range[0]} → {date_range[1]}")

    if score_col:
        score_stats = con.execute(
            f"SELECT MIN({score_col}), MAX({score_col}), AVG({score_col}), "
            f"PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY {score_col}) "
            f"FROM {table_name} WHERE {score_col} IS NOT NULL"
        ).fetchone()
        print(f"  {score_col}: min={score_stats[0]:.1f}, max={score_stats[1]:.1f}, "
              f"avg={score_stats[2]:.2f}, median={score_stats[3]:.1f}")

print("validate_table() defined")

validate_table() defined


---
## Section 1 — Contact Final

Merge `contact_clean` with `Data/contact-addition.csv` (same schema), deduplicate on `developer_id` keeping the most recently modified record, and produce `contact_final`.

Contact is cleaned first because activity and SDK rows join against `developer_id`.

In [5]:
# Load contact-addition.csv → contact_supplement_raw
con.execute("""
CREATE OR REPLACE TABLE contact_supplement_raw AS
SELECT *
FROM read_csv_auto(
    'Data/contact-addition.csv',
    header = True,
    delim = ',',
    all_varchar = True
)
""")

count = con.execute("SELECT COUNT(*) FROM contact_supplement_raw").fetchone()[0]
print(f"contact_supplement_raw loaded: {count:,} rows")
display(con.execute("SELECT * FROM contact_supplement_raw LIMIT 3").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

contact_supplement_raw loaded: 478,293 rows


,developer_id,program_application_source,country,region,sub_region,zone,territory,organization_english_name,development_areas,other_development_areas,industry_segment_vertical,other_industry_segment_vertical,sub_industry_segment_vertical,fields_of_interest,other_fields_of_interest,first_program_application_date,account_id,account_name,last_activity_date,last_modified_date,created_date,wwfo_category,wwfo_target_list,account_industry_segment,account_source,account_type,devzone_last_login_date,organization_website,inception_id,first_activity_date,normalized_account_name,rdp_exit_date
0,09cd31ebdf759811e55c4d47eaba9c6f3aba2bff9c9027...,devzone,China,APAC,China,China,None,ZJU,Data Science;conversational_ai;Computer Vision...,None,AEC,NaN,None,None,None,2018-09-05T00:00:00.000Z,31153,Not Normalized,2026-01-17T02:03:43.000Z,2026-01-17T02:27:17.249Z,2018-10-19T22:24:18.000Z,None,None,None,Manual,None,2026-01-16T18:03:43.000Z,None,None,2026-01-17T02:03:43.000Z,Not Normalized,None
1,09eeefb8e68192e5806a9516311e99f9c56dabdb963c72...,devzone,China,APAC,China,China,None,-,Agentic AI / Generative AI;Computer Vision / V...,None,Other,software,None,None,None,2018-05-27T00:00:00.000Z,31153,Not Normalized,2026-03-07T16:47:30.000Z,2026-03-07T16:57:17.078Z,2018-10-20T23:30:17.000Z,None,None,None,Manual,None,2026-03-07T08:47:30.000Z,None,None,2026-03-07T16:47:30.000Z,Not Normalized,None
2,0a5528c47e6d5491649bfff946421e2220d07de65c7f3c...,devzone,China,APAC,China,China,None,SX,Data Science;conversational_ai;Computer Vision...,None,Other,Software,None,None,None,2010-08-18T00:00:00.000Z,31153,Not Normalized,2025-10-02T17:12:28.000Z,2025-10-02T17:27:17.287Z,2013-04-18T00:42:34.000Z,None,None,None,Manual,None,2025-10-02T10:12:28.000Z,None,None,2025-10-02T17:12:28.000Z,Not Normalized,None


In [6]:
# Standardize supplement using same logic as contact_clean
con.execute("""
CREATE OR REPLACE TABLE contact_supplement_clean AS
SELECT
    TRIM(developer_id)                                                          AS developer_id,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(program_application_source, 'unknown'), '\\s+', ' '))) AS program_application_source,
    UPPER(TRIM(country))                                                        AS country,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(region,       'unknown'), '\\s+', ' '))) AS region,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(sub_region,   'unknown'), '\\s+', ' '))) AS sub_region,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(zone,         'unknown'), '\\s+', ' '))) AS zone,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(territory,    'unknown'), '\\s+', ' '))) AS territory,
    TRIM(organization_english_name)                                             AS organization_english_name,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(development_areas,            'unknown'), '\\s+', ' '))) AS development_areas,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(other_development_areas,      'unknown'), '\\s+', ' '))) AS other_development_areas,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(industry_segment_vertical,    'unknown'), '\\s+', ' '))) AS industry_segment_vertical,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(other_industry_segment_vertical, 'unknown'), '\\s+', ' '))) AS other_industry_segment_vertical,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(sub_industry_segment_vertical,'unknown'), '\\s+', ' '))) AS sub_industry_segment_vertical,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(fields_of_interest,           'unknown'), '\\s+', ' '))) AS fields_of_interest,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(other_fields_of_interest,     'unknown'), '\\s+', ' '))) AS other_fields_of_interest,
    TRY_CAST(first_program_application_date AS TIMESTAMP)                       AS first_program_application_date,
    account_id,
    TRIM(account_name)                                                          AS account_name,
    TRY_CAST(last_activity_date   AS TIMESTAMP)                                 AS last_activity_date,
    CASE WHEN TRY_CAST(last_modified_date AS TIMESTAMP) > CURRENT_TIMESTAMP
         THEN NULL
         ELSE TRY_CAST(last_modified_date AS TIMESTAMP) END                     AS last_modified_date,
    CASE WHEN TRY_CAST(created_date AS TIMESTAMP) > CURRENT_TIMESTAMP
         THEN NULL
         ELSE TRY_CAST(created_date AS TIMESTAMP) END                           AS created_date,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(wwfo_category,           'unknown'), '\\s+', ' '))) AS wwfo_category,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(wwfo_target_list,        'unknown'), '\\s+', ' '))) AS wwfo_target_list,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(account_industry_segment,'unknown'), '\\s+', ' '))) AS account_industry_segment,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(account_source,          'unknown'), '\\s+', ' '))) AS account_source,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(account_type,            'unknown'), '\\s+', ' '))) AS account_type,
    CASE WHEN TRY_CAST(devzone_last_login_date AS TIMESTAMP) > CURRENT_TIMESTAMP
         THEN NULL
         ELSE TRY_CAST(devzone_last_login_date AS TIMESTAMP) END                AS devzone_last_login_date,
    TRIM(organization_website)                                                  AS organization_website,
    inception_id,
    CASE WHEN TRY_CAST(first_activity_date AS TIMESTAMP) > CURRENT_TIMESTAMP
         THEN NULL
         ELSE TRY_CAST(first_activity_date AS TIMESTAMP) END                    AS first_activity_date,
    TRIM(normalized_account_name)                                               AS normalized_account_name,
    CASE WHEN TRY_CAST(rdp_exit_date AS TIMESTAMP) > CURRENT_TIMESTAMP
         THEN NULL
         ELSE TRY_CAST(rdp_exit_date AS TIMESTAMP) END                          AS rdp_exit_date
FROM contact_supplement_raw
WHERE developer_id IS NOT NULL
  AND TRIM(developer_id) != ''
""")

count = con.execute("SELECT COUNT(*) FROM contact_supplement_clean").fetchone()[0]
print(f"contact_supplement_clean: {count:,} rows")

contact_supplement_clean: 478,293 rows


In [7]:
# UNION contact_clean + contact_supplement_clean,
# deduplicate on developer_id keeping the row with the latest last_modified_date
con.execute("""
CREATE OR REPLACE TABLE contact_final AS
WITH combined AS (
    SELECT * FROM contact_clean
    UNION ALL
    SELECT * FROM contact_supplement_clean
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY developer_id
               ORDER BY last_modified_date DESC NULLS LAST
           ) AS rn
    FROM combined
)
SELECT * EXCLUDE (rn)
FROM ranked
WHERE rn = 1
""")

validate_table(con, "contact_final", "developer_id")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Table: contact_final
  Rows          : 9,381,490
  Null developer_id   : 0 (0.00%)
  Duplicates on developer_id: 0


---
## Section 2 — Activity Final

Clean `activity_clean` (69.3M rows) in four stages:
1. Normalize text fields, filter invalid rows
2. Cap and fill scores (median fallback — `Activity_Score_Mapping.csv` unavailable)
3. Deduplicate
4. Write `activity_final`

In [8]:
# Stage 1: normalize text, filter invalid rows → activity_base
con.execute("""
CREATE OR REPLACE TABLE activity_base AS
SELECT
    TRIM(dev_contact)                                                                       AS dev_contact,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(activity,           'unknown'), '\\s+', ' ')))       AS activity,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(activity_name,      'unknown'), '\\s+', ' ')))       AS activity_name,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(activity_type,      'unknown'), '\\s+', ' ')))       AS activity_type,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(activity_role,      'unknown'), '\\s+', ' ')))       AS activity_role,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(activity_attendance,'unknown'), '\\s+', ' ')))       AS activity_attendance,
    activity_score,
    activity_date,
    NULLIF(TRIM(activity_id), '')                                                           AS activity_id,
    filepath,
    pk1,
    pk2,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(lead_source,        'unknown'), '\\s+', ' ')))       AS lead_source,
    nvidia_campaign_id,
    gtc_nvidia_campaign_id,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(lead_source_details,'unknown'), '\\s+', ' ')))       AS lead_source_details
FROM activity_clean
WHERE dev_contact IS NOT NULL
  AND TRIM(dev_contact) != ''
  AND activity_date IS NOT NULL
  AND activity_date <= CURRENT_DATE
""")

count = con.execute("SELECT COUNT(*) FROM activity_base").fetchone()[0]
print(f"activity_base (after filtering): {count:,} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

activity_base (after filtering): 69,347,501 rows


In [9]:
# Stage 2: compute median score, then cap outliers and fill NULLs → activity_work
# NOTE: Activity_Score_Mapping.csv is unavailable; median is used as the sole fallback.
#       If the mapping file becomes available, add a LEFT JOIN here before the COALESCE.

median_score = con.execute(
    "SELECT PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY activity_score) "
    "FROM activity_base WHERE activity_score IS NOT NULL"
).fetchone()[0]

print(f"Median activity_score (non-null): {median_score}")

con.execute(f"""
CREATE OR REPLACE TABLE activity_work AS
SELECT
    dev_contact,
    activity,
    activity_name,
    activity_type,
    activity_role,
    activity_attendance,
    GREATEST(0.0, LEAST(100.0, COALESCE(activity_score, {median_score}))) AS activity_score,
    activity_date,
    activity_id,
    filepath,
    pk1,
    pk2,
    lead_source,
    nvidia_campaign_id,
    gtc_nvidia_campaign_id,
    lead_source_details
FROM activity_base
""")

print(f"activity_work created")

# Confirm score range
score_range = con.execute(
    "SELECT MIN(activity_score), MAX(activity_score) FROM activity_work"
).fetchone()
print(f"Score range: [{score_range[0]}, {score_range[1]}]")

Median activity_score (non-null): 3.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

activity_work created
Score range: [0.0, 100.0]


In [10]:
# Stage 3: deduplicate
# - Rows WITH activity_id: deduplicate on activity_id
# - Rows WITHOUT activity_id: deduplicate on composite key

con.execute("""
CREATE OR REPLACE TABLE activity_final AS
WITH with_id AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY activity_id ORDER BY activity_date DESC) AS rn
    FROM activity_work
    WHERE activity_id IS NOT NULL
),
without_id AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY dev_contact, activity_date, activity_name,
                            activity_type, activity_role, activity_attendance
               ORDER BY activity_score DESC
           ) AS rn
    FROM activity_work
    WHERE activity_id IS NULL
)
SELECT * EXCLUDE (rn) FROM with_id    WHERE rn = 1
UNION ALL
SELECT * EXCLUDE (rn) FROM without_id WHERE rn = 1
""")

validate_table(con, "activity_final", "dev_contact",
               date_col="activity_date", score_col="activity_score")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Table: activity_final
  Rows          : 68,314,434
  Null dev_contact    : 0 (0.00%)
  Duplicates on dev_contact: 60,660,007
  activity_date range: 2020-01-01 00:00:00 → 2026-03-12 17:24:45


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  activity_score: min=0.0, max=100.0, avg=3.33, median=3.0


---
## Section 3 — SDK Download Final

Clean `sdk_download_clean` (93M rows): canonicalize source values, fix negative download counts, filter invalid dates, deduplicate.

In [11]:
# Stage 1: normalize, fix negatives, filter → sdk_work
con.execute("""
CREATE OR REPLACE TABLE sdk_work AS
SELECT
    CASE
        WHEN LOWER(TRIM(source)) IN ('pypi', 'py pi', 'python package index') THEN 'pypi'
        WHEN LOWER(TRIM(source)) IN ('github', 'git hub')                     THEN 'github'
        WHEN LOWER(TRIM(source)) IN ('nvidia', 'developer.nvidia',
                                     'developer portal', 'nvidia developer')   THEN 'nvidia'
        ELSE LOWER(TRIM(REGEXP_REPLACE(COALESCE(source, 'unknown'), '\\s+', ' ')))
    END                                                                             AS source,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(sdk_name,        'unknown'), '\\s+', ' '))) AS sdk_name,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(product_name,    'unknown'), '\\s+', ' '))) AS product_name,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(product_release, 'unknown'), '\\s+', ' '))) AS product_release,
    UPPER(TRIM(country))                                                            AS country,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(region,          'unknown'), '\\s+', ' '))) AS region,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(subregion,       'unknown'), '\\s+', ' '))) AS subregion,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(territory,       'unknown'), '\\s+', ' '))) AS territory,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(zone,            'unknown'), '\\s+', ' '))) AS zone,
    download_date,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(file_type,         'unknown'), '\\s+', ' '))) AS file_type,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(operating_system,  'unknown'), '\\s+', ' '))) AS operating_system,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(os_distribution,   'unknown'), '\\s+', ' '))) AS os_distribution,
    LOWER(TRIM(REGEXP_REPLACE(COALESCE(architecture,      'unknown'), '\\s+', ' '))) AS architecture,
    COALESCE(kpi, 0.0)              AS kpi,
    GREATEST(download_count, 0)     AS download_count
FROM sdk_download_clean
WHERE download_date IS NOT NULL
  AND download_date <= CURRENT_DATE
""")

count = con.execute("SELECT COUNT(*) FROM sdk_work").fetchone()[0]
print(f"sdk_work (after filtering): {count:,} rows")

neg = con.execute("SELECT COUNT(*) FROM sdk_work WHERE download_count < 0").fetchone()[0]
print(f"Negative download_count remaining: {neg}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sdk_work (after filtering): 93,038,213 rows
Negative download_count remaining: 0


In [12]:
# Stage 2: deduplicate on full row composite key → sdk_download_final
con.execute("""
CREATE OR REPLACE TABLE sdk_download_final AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY source, sdk_name, product_name, product_release,
                            country, region, subregion, territory, zone,
                            download_date, file_type, operating_system,
                            os_distribution, architecture
               ORDER BY download_count DESC
           ) AS rn
    FROM sdk_work
)
SELECT * EXCLUDE (rn)
FROM ranked
WHERE rn = 1
""")

validate_table(con, "sdk_download_final", "sdk_name", date_col="download_date")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Table: sdk_download_final
  Rows          : 91,752,304
  Null sdk_name       : 0 (0.00%)
  Duplicates on sdk_name: 91,752,118
  download_date range: 2020-01-01 → 2026-03-12


---
## Section 4 — Post-Cleaning Validation Summary

In [13]:
# Row counts for all final tables
tables = ["contact_final", "activity_final", "sdk_download_final"]
print("Final table row counts:")
for t in tables:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<25}: {n:,}")

Final table row counts:
  contact_final            : 9,381,490
  activity_final           : 68,314,434
  sdk_download_final       : 91,752,304


In [14]:
# Activity rows that don't match any contact (should be very small)
unmatched = con.execute("""
SELECT COUNT(*) AS unmatched_activity_rows
FROM activity_final a
LEFT JOIN contact_final c ON a.dev_contact = c.developer_id
WHERE c.developer_id IS NULL
""").fetchone()[0]

total_activity = con.execute("SELECT COUNT(*) FROM activity_final").fetchone()[0]
pct = unmatched / total_activity * 100
print(f"Unmatched activity rows: {unmatched:,} ({pct:.4f}% of activity_final)")

# Contacts without any activity
no_activity = con.execute("""
SELECT COUNT(*) AS contacts_without_activity
FROM contact_final c
LEFT JOIN activity_final a ON c.developer_id = a.dev_contact
WHERE a.dev_contact IS NULL
""").fetchone()[0]
total_contact = con.execute("SELECT COUNT(*) FROM contact_final").fetchone()[0]
print(f"Contacts without activity: {no_activity:,} ({no_activity/total_contact*100:.1f}% of contact_final)")

Unmatched activity rows: 534 (0.0008% of activity_final)
Contacts without activity: 1,727,080 (18.4% of contact_final)


---
## Section 5 — Sampling

Create reproducible 100k random samples of each final table, stored in DuckDB and exported to `Data/`.

In [15]:
SEED = 42
SAMPLE_SIZE = 100_000

samples = {
    "activity_sample":     ("activity_final",      "Data/activity_sample.csv"),
    "contact_sample":      ("contact_final",        "Data/contact_sample.csv"),
    "sdk_download_sample": ("sdk_download_final",   "Data/sdk_download_sample.csv"),
}

for sample_table, (source_table, csv_path) in samples.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {sample_table} AS
        SELECT *
        FROM {source_table}
        USING SAMPLE {SAMPLE_SIZE} ROWS (reservoir, {SEED})
    """)
    n = con.execute(f"SELECT COUNT(*) FROM {sample_table}").fetchone()[0]

    con.execute(f"""
        COPY {sample_table} TO '{csv_path}' (HEADER, DELIMITER ',')
    """)
    print(f"{sample_table}: {n:,} rows → {csv_path}")

print("\nAll samples created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

activity_sample: 100,000 rows → Data/activity_sample.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

contact_sample: 100,000 rows → Data/contact_sample.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

sdk_download_sample: 100,000 rows → Data/sdk_download_sample.csv

All samples created.


In [16]:
con.close()
print("Connection closed. Cleaning pipeline complete.")
print("\nNext step: run CleanDataSanity.ipynb to validate pre-join integrity.")

Connection closed. Cleaning pipeline complete.

Next step: run CleanDataSanity.ipynb to validate pre-join integrity.
